# recs_018 — D6 rank-only spikes (frozen retrieve trunk)

**D6a:** rank head on **frozen** `two_tower_v1` features → saves `rank_head.keras` only.

**D6b:** **separate** rank bi-encoder (warm-start from retrieve, then diverge) → saves `rank_biencoder.keras`.

**Hard rule:** `artifacts/recs/towers/.../two_tower_v1.keras` is **never** overwritten.

**Promotion bar:** beat D1 `heuristic_logpop_blend` on val NDCG@10 overall + slice A.

Plan: [`docs/ranker_exploration_plan.md`](../../docs/ranker_exploration_plan.md) § D6.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from steam_review_ml.evaluation.example_cohort import load_retrieval_pool_rows, load_retrieval_pools_jsonl
from steam_review_ml.evaluation.heuristic_ranker import (
    METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND,
    pool_rerank_registry,
    rerank_scores_on_pool,
)
from steam_review_ml.evaluation.retrieval_offline_eval import (
    _oracle_ranked_indices_from_retrieved,
    _rank_rows,
    _table_personalization,
    average_precision_at_k,
    hit_rate_at_k,
    load_eval_examples_from_parquet,
    mrr,
    ndcg_at_k,
    prepare_eval_inputs_from_cache,
)
from steam_review_ml.recommender.ranker_d6_biencoder import (
    build_rank_biencoder_model,
    make_pool_score_fn as make_d6b_pool_score_fn,
    save_rank_biencoder_bundle,
    train_rank_biencoder,
)
from steam_review_ml.recommender.ranker_d6_common import (
    D6ListwiseConfig,
    attach_query_text,
    query_text_by_ex_idx,
)
from steam_review_ml.recommender.ranker_d6_rank_head import (
    load_frozen_trunk,
    make_pool_score_fn as make_d6a_pool_score_fn,
    save_rank_head_bundle,
    train_rank_head_ranker,
)
from steam_review_ml.recommender.two_tower_train import load_hub_settings

K_FINAL = 10
K_PERSONALIZATION = 10
POOL_METHOD = "two_tower_v1"
TUNE_FRAC = 0.1
SPLIT_SEED = 2027
SMOKE = False  # set False for full train_ranker_v1 run
SMOKE_N = 512

TRAIN_POOLS_PARQUET = REPO_ROOT / "artifacts/recs/ranker_pools/train_ranker_v1/two_tower_v1.parquet"
TRAIN_COHORT_PARQUET = REPO_ROOT / "artifacts/recs/eval_cache/train_ranker_v1/example_cohort.parquet"
VAL_JSONL = REPO_ROOT / "artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl"
VAL_COHORT_PARQUET = REPO_ROOT / "artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet"
RETRIEVE_MODEL_PATH = REPO_ROOT / "artifacts/recs/towers/val_dev_12k_v1/updated_user__updated_profile200_item.keras"
D6A_OUT = REPO_ROOT / "artifacts/recs/rankers/d6_rank_head_v1"
D6B_OUT = REPO_ROOT / "artifacts/recs/rankers/d6_rank_biencoder_v1"

cfg = D6ListwiseConfig(epochs=3 if SMOKE else 15, list_batch_size=32 if SMOKE else 64)
print(f"SMOKE={SMOKE} cfg={cfg}")

/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-14 09:22:36.917030: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-14 09:22:36.953878: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781443356.986153   65014 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781443356.997073   65014 cuda_bl

SMOKE=False cfg=D6ListwiseConfig(hidden_units=32, dropout=0.1, learning_rate=0.001, list_batch_size=64, max_list_len=100, epochs=15, early_stopping_patience=3, random_seed=2028, user_encode_batch=64)


In [2]:
train_pools = load_retrieval_pool_rows(TRAIN_POOLS_PARQUET)
train_q = query_text_by_ex_idx(TRAIN_COHORT_PARQUET)
train_pools = attach_query_text(train_pools, train_q)

val_q = query_text_by_ex_idx(VAL_COHORT_PARQUET)
val_pools = attach_query_text(load_retrieval_pools_jsonl(VAL_JSONL, method=POOL_METHOD), val_q)
val_examples = load_eval_examples_from_parquet(VAL_COHORT_PARQUET)
for i, ex in enumerate(val_examples):
    ex["ex_idx"] = i

inputs = prepare_eval_inputs_from_cache(
    repo_root=REPO_ROOT,
    split="val",
    min_review_chars=30,
    examples_parquet=VAL_COHORT_PARQUET,
    artifact_dir=REPO_ROOT / "artifacts/recs",
    verbose=False,
)
app_ids = inputs.app_ids
app_to_row = inputs.app_to_row
pop_row = inputs.pop_row

if SMOKE:
    train_pools = train_pools[:SMOKE_N]

print(f"train pools={len(train_pools):,} val pools={len(val_pools):,} catalog={len(app_ids)}")

train pools=51,691 val pools=12,500 catalog=315


In [3]:
def stratified_ex_idx_split(pools, *, tune_frac: float, seed: int):
    by_slice: dict[str, list[int]] = {}
    for row in pools:
        by_slice.setdefault(str(row["slice_name"]), []).append(int(row["ex_idx"]))
    rng = np.random.default_rng(seed)
    tune, fit = set(), set()
    for ex_idxs in by_slice.values():
        ex_idxs = list(ex_idxs)
        rng.shuffle(ex_idxs)
        n_tune = max(1, int(round(len(ex_idxs) * tune_frac)))
        tune.update(ex_idxs[:n_tune])
        fit.update(ex_idxs[n_tune:])
    return fit, tune


fit_idx, tune_idx = stratified_ex_idx_split(train_pools, tune_frac=TUNE_FRAC, seed=SPLIT_SEED)
train_fit = [r for r in train_pools if int(r["ex_idx"]) in fit_idx]
train_tune = [r for r in train_pools if int(r["ex_idx"]) in tune_idx]
print(f"train_fit={len(train_fit):,} train_tune={len(train_tune):,}")

train_fit=46,522 train_tune=5,169


In [4]:
def pool_scores_to_ranked_indices(pool_app_ids, pool_scores, *, app_to_row, k_final=K_FINAL):
    order = sorted(range(len(pool_app_ids)), key=lambda i: (-float(pool_scores[i]), int(pool_app_ids[i])))
    return [int(app_to_row[int(pool_app_ids[i])]) for i in order[:k_final]]


def popularity_catalog_scores(*, query_app_id: int) -> np.ndarray:
    s = np.asarray(pop_row, dtype=np.float64).copy()
    row = app_to_row.get(int(query_app_id))
    if row is not None:
        s[row] = -np.inf
    return s


def full_catalog_scores_for_pool_row(
    row: dict[str, Any],
    *,
    score_fn: Callable[..., np.ndarray] | None = None,
    catalog_pop: bool = False,
) -> np.ndarray:
    """Full-catalog scores for personalization: only the frozen pool gets rerank scores."""
    if catalog_pop:
        return popularity_catalog_scores(query_app_id=int(row["query_app_id"]))
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    if score_fn is None:
        pool_scores = np.asarray(ret_sc, dtype=np.float64)
    else:
        pool_scores = np.asarray(
            score_fn(
                pool_apps,
                ret_sc,
                query_text=str(row["query_text"]),
                pop_row=pop_row,
                app_to_row=app_to_row,
            ),
            dtype=np.float64,
        )
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, score in zip(pool_apps, pool_scores):
        full[int(app_to_row[int(app_id)])] = float(score)
    return full


def eval_pool_row(row, *, method: str, score_fn=None, params=None, oracle=False, catalog_pop=False):
    positives = {int(x) for x in json.loads(row["validation_positive_app_ids_json"])}
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    retrieved_rows = np.asarray([app_to_row[a] for a in pool_apps], dtype=np.int64)
    if oracle:
        ranked = _oracle_ranked_indices_from_retrieved(retrieved_rows, positives, app_ids)[:K_FINAL]
    elif catalog_pop:
        full = pop_row.copy()
        qrow = app_to_row.get(int(row["query_app_id"]))
        if qrow is not None:
            full[int(qrow)] = -np.inf
        ranked = [int(i) for i in _rank_rows(full)[:K_FINAL]]
    elif score_fn is None:
        ranked = pool_scores_to_ranked_indices(pool_apps, np.asarray(ret_sc), app_to_row=app_to_row)
    else:
        blend = score_fn(pool_apps, ret_sc, query_text=str(row["query_text"]), **(params or {}), pop_row=pop_row, app_to_row=app_to_row)
        ranked = pool_scores_to_ranked_indices(pool_apps, blend, app_to_row=app_to_row)
    ranked = np.asarray(ranked, dtype=np.int64)
    return {
        "method": method,
        "slice_name": row.get("slice_name", ""),
        "NDCG@K": ndcg_at_k(ranked, positives, K_FINAL, app_ids),
        "Hit@K": hit_rate_at_k(ranked, positives, K_FINAL, app_ids),
        "MAP@K": average_precision_at_k(ranked, positives, K_FINAL, app_ids),
        "MRR": mrr(ranked, positives, app_ids),
    }


def summarize(rows: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    overall = df.groupby("method")[["NDCG@K", "Hit@K", "MAP@K", "MRR"]].mean().reset_index()
    slice_a = df[df["slice_name"] == "slice_a_multi_target"].groupby("method")[["NDCG@K"]].mean().reset_index()
    slice_a = slice_a.rename(columns={"NDCG@K": "NDCG@K_slice_a"})
    return overall.merge(slice_a, on="method", how="left").sort_values("NDCG@K", ascending=False)

## D6a — rank head on frozen trunk

In [5]:
ctx = load_frozen_trunk(RETRIEVE_MODEL_PATH, inputs.retriever)
d6a_head, d6a_hist = train_rank_head_ranker(
    fit_pools=train_fit,
    tune_pools=train_tune,
    ctx=ctx,
    app_ids=app_ids,
    app_to_row=app_to_row,
    cfg=cfg,
    k_final=K_FINAL,
)
display(d6a_hist.tail())
save_rank_head_bundle(
    head_model=d6a_head,
    output_dir=D6A_OUT,
    retrieve_model_path=RETRIEVE_MODEL_PATH,
    projection_dim=ctx.projection_dim,
    cfg=cfg,
)
d6a_score_fn = make_d6a_pool_score_fn(d6a_head, ctx, app_to_row=app_to_row, max_list_len=cfg.max_list_len)

/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
I0000 00:00:1781443374.602022   65014 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5199 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5070 Laptop GPU, pci bus id: 0000:02:00.0, compute capability: 12.0
2026-06-14 09:22:58.877845: E tensorflow/core/util/util.cc:131] oneDNN supports DT_INT64 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
I0000 00:00:1781443380.716951   65282 service.cc:152] XLA service 0x7669140145c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:0

D6a epoch 1/15: train_loss=4.2039 tune_NDCG@10=0.1649
D6a epoch 2/15: train_loss=3.9750 tune_NDCG@10=0.1703
D6a epoch 3/15: train_loss=3.9563 tune_NDCG@10=0.1702
D6a epoch 4/15: train_loss=3.9491 tune_NDCG@10=0.1698
D6a epoch 5/15: train_loss=3.9414 tune_NDCG@10=0.1704
D6a epoch 6/15: train_loss=3.9376 tune_NDCG@10=0.1709
D6a epoch 7/15: train_loss=3.9336 tune_NDCG@10=0.1725
D6a epoch 8/15: train_loss=3.9333 tune_NDCG@10=0.1711
D6a epoch 9/15: train_loss=3.9304 tune_NDCG@10=0.1725
D6a epoch 10/15: train_loss=3.9263 tune_NDCG@10=0.1723


,epoch,train_loss,tune_NDCG
5,6,3.937584,0.170866
6,7,3.933600,0.172522
7,8,3.933287,0.171084
8,9,3.930422,0.172502
9,10,3.926307,0.172273


## D6b — separate rank bi-encoder

In [6]:
import gc
import tensorflow as tf
del ctx, d6a_head
tf.keras.backend.clear_session()
gc.collect()

0

In [7]:

hub_url, hub_max_chars = load_hub_settings(inputs.retriever)
d6b_model = build_rank_biencoder_model(
    inputs.retriever,
    init_from_retrieve_path=RETRIEVE_MODEL_PATH,
    hub_url=hub_url,
)
d6b_model, d6b_hist = train_rank_biencoder(
    fit_pools=train_fit,
    tune_pools=train_tune,
    rank_model=d6b_model,
    app_ids=app_ids,
    app_to_row=app_to_row,
    max_chars=hub_max_chars,
    cfg=cfg,
    k_final=K_FINAL,
)
display(d6b_hist.tail())
save_rank_biencoder_bundle(
    rank_model=d6b_model,
    output_dir=D6B_OUT,
    retrieve_model_path=RETRIEVE_MODEL_PATH,
    cfg=cfg,
)
d6b_score_fn = make_d6b_pool_score_fn(d6b_model, app_to_row=app_to_row, max_chars=hub_max_chars)

D6b epoch 1/15: train_loss=13.0254 tune_NDCG@10=0.1247
D6b epoch 2/15: train_loss=5.2554 tune_NDCG@10=0.1302
D6b epoch 3/15: train_loss=4.2392 tune_NDCG@10=0.1398
D6b epoch 4/15: train_loss=4.0533 tune_NDCG@10=0.1523
D6b epoch 5/15: train_loss=3.9902 tune_NDCG@10=0.1581
D6b epoch 6/15: train_loss=3.9571 tune_NDCG@10=0.1597
D6b epoch 7/15: train_loss=3.9254 tune_NDCG@10=0.1603
D6b epoch 8/15: train_loss=3.9009 tune_NDCG@10=0.1603
D6b epoch 9/15: train_loss=3.8778 tune_NDCG@10=0.1628
D6b epoch 10/15: train_loss=3.8667 tune_NDCG@10=0.1637
D6b epoch 11/15: train_loss=3.8518 tune_NDCG@10=0.1583
D6b epoch 12/15: train_loss=3.8390 tune_NDCG@10=0.1520
D6b epoch 13/15: train_loss=3.8199 tune_NDCG@10=0.1538


,epoch,train_loss,tune_NDCG
8,9,3.877800,0.162797
9,10,3.866683,0.163677
10,11,3.851780,0.158268
11,12,3.839018,0.151976
12,13,3.819882,0.153788


## Val head-to-head vs D1 (+ personalization guardrails)

In [8]:
d1_spec = pool_rerank_registry()[METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND]


def eval_d1(row):
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    blend = rerank_scores_on_pool(pool_apps, ret_sc, d1_spec, pop_row=pop_row, app_to_row=app_to_row)
    return eval_pool_row(row, method="D1_heuristic_logpop_blend", score_fn=lambda apps, retr, **kw: blend)


rows = []
for row in val_pools:
    rows.append(eval_pool_row(row, method="two_tower_v1_bare", score_fn=None))
    rows.append(eval_d1(row))
    rows.append(eval_pool_row(row, method="D6a_rank_head", score_fn=d6a_score_fn))
    rows.append(eval_pool_row(row, method="D6b_rank_biencoder", score_fn=d6b_score_fn))

summary = summarize(rows)
display(summary)
d1_overall = float(summary.loc[summary["method"] == "D1_heuristic_logpop_blend", "NDCG@K"].iloc[0])
print(f"D1 overall NDCG@10 = {d1_overall:.4f} (promotion bar)")

pools_by_ex = {int(r["ex_idx"]): r for r in val_pools}
pool_ex_indices = set(pools_by_ex.keys())


def _make_pool_catalog_scorer(
    score_fn: Callable[..., np.ndarray] | None = None,
) -> Callable[[dict[str, Any]], np.ndarray]:
    def scorer(ex: dict[str, Any]) -> np.ndarray:
        return full_catalog_scores_for_pool_row(
            pools_by_ex[int(ex["ex_idx"])], score_fn=score_fn
        )

    return scorer


def _d1_pool_scores(pool_apps, ret_sc, **kw):
    return rerank_scores_on_pool(pool_apps, ret_sc, d1_spec, pop_row=pop_row, app_to_row=app_to_row)


person_methods: dict[str, Callable[[dict[str, Any]], np.ndarray]] = {
    "popularity_train": lambda ex: popularity_catalog_scores(query_app_id=int(ex["query_app_id"])),
    "two_tower_v1_bare": _make_pool_catalog_scorer(score_fn=None),
    "D1_heuristic_logpop_blend": _make_pool_catalog_scorer(score_fn=_d1_pool_scores),
    "D6a_rank_head": _make_pool_catalog_scorer(score_fn=d6a_score_fn),
    "D6b_rank_biencoder": _make_pool_catalog_scorer(score_fn=d6b_score_fn),
}

person_cols = [
    f"ILD@{K_PERSONALIZATION}",
    f"CatalogCoverage@{K_PERSONALIZATION}",
    f"Novelty@{K_PERSONALIZATION}",
    f"PersonalizationGapVsPopularity@{K_PERSONALIZATION}",
]

print(
    f"\nPersonalization guardrails @ k={K_PERSONALIZATION} (non-gating; vs popularity_train baseline):\n"
    f"  ILD@{K_PERSONALIZATION} — embedding-space list diversity (higher = more spread)\n"
    f"  CatalogCoverage@{K_PERSONALIZATION} — share of catalog appearing in any top-{K_PERSONALIZATION} list\n"
    f"  Novelty@{K_PERSONALIZATION} — mean -log2(train-pop share); higher = less head-item heavy\n"
    f"  PersonalizationGapVsPopularity@{K_PERSONALIZATION} — 1 - Jaccard vs pop-only top-{K_PERSONALIZATION} "
    f"(higher = less pop-like; D1 ~0.72, bare two-tower ~0.99 on val)"
)

df_person = _table_personalization(
    methods=person_methods,
    examples=val_examples,
    X=inputs.retriever.embedding_matrix,
    app_ids=app_ids,
    pop_row=pop_row,
    k_personalization=K_PERSONALIZATION,
    example_indices=pool_ex_indices,
)
summary_full = summary.merge(df_person, on="method", how="left")
print(f"personalization methods: {len(df_person)}")
print(summary_full[["method", "NDCG@K"] + person_cols].to_string(index=False))
display(summary_full.sort_values("NDCG@K", ascending=False))

print(
    """
recs_018 — errors hit so far
----------------------------
1) load_frozen_trunk: Dense has no output_shape (D6a cell)
   Meaning: Keras 3 Dense layers are not built until first call, so .output_shape is missing.
   Why it failed: code read projection dim from an unbuilt layer.
   Fix: use model._projection_dim (or item_vectors.shape[1]). Already in ranker_d6_rank_head.py.

2) build_rank_biencoder_model: "expecting one weight" (D6b cell)
   Meaning: set_weights got a full checkpoint but the new model only had one weight slot.
   Why it failed: warm-start ran before build(); only item_base_embeddings existed.
   Fix: call rank_model.build() before set_weights(). Already in ranker_d6_biencoder.py.

3) D6b GPU OOM after D6a (D6b cell)
   Meaning: GPU ran out of memory (~5 GB already in use).
   Why it failed: D6a leaves the full retrieve model in ctx on GPU; D6b loads a second copy (+ brief third during warm-start).
   Fix: before D6b, run `del ctx, d6a_head; tf.keras.backend.clear_session(); gc.collect()` then re-run D6a after D6b if you need both for eval — or run D6b before D6a, or restart kernel between them.

4) eval cell IndexError: index 875210 out of bounds for size 315
   Meaning: ndcg_at_k expects catalog row indices (0..314), not Steam app_ids.
   Why it failed: pool_scores_to_ranked_indices returned app_ids; ndcg_at_k did app_ids[875210].
   Fix: map ranked outputs through app_to_row before metrics. Fixed in eval_pool_row helper cell.

5) eval cell AttributeError: 'list' object has no attribute 'tolist'
   Meaning: average_precision_at_k / mrr expect a numpy array, not a Python list.
   Why it failed: pool_scores_to_ranked_indices returns a list; ndcg_at_k accepts lists but MAP@K does not.
   Fix: wrap ranked with np.asarray(..., dtype=np.int64) before calling metrics.
"""
)

,method,NDCG@K,Hit@K,MAP@K,MRR,NDCG@K_slice_a
0,D1_heuristic_logpop_blend,0.092892,0.19328,0.064321,0.067059,0.068322
1,D6a_rank_head,0.079514,0.16544,0.054695,0.056830,0.052844
2,D6b_rank_biencoder,0.077646,0.16144,0.053708,0.055946,0.054507
3,two_tower_v1_bare,0.018161,0.04680,0.010325,0.011008,0.020537


D1 overall NDCG@10 = 0.0929 (promotion bar)

Personalization guardrails @ k=10 (non-gating; vs popularity_train baseline):
  ILD@10 — embedding-space list diversity (higher = more spread)
  CatalogCoverage@10 — share of catalog appearing in any top-10 list
  Novelty@10 — mean -log2(train-pop share); higher = less head-item heavy
  PersonalizationGapVsPopularity@10 — 1 - Jaccard vs pop-only top-10 (higher = less pop-like; D1 ~0.72, bare two-tower ~0.99 on val)
personalization methods: 5
                   method   NDCG@K   ILD@10  CatalogCoverage@10  Novelty@10  PersonalizationGapVsPopularity@10
D1_heuristic_logpop_blend 0.092892 0.204632            0.501587    6.272161                           0.720097
            D6a_rank_head 0.079514 0.208822            0.457143    6.320686                           0.732713
       D6b_rank_biencoder 0.077646 0.211762            0.787302    6.508531                           0.766363
        two_tower_v1_bare 0.018161 0.261639            0.987302  

,method,NDCG@K,Hit@K,MAP@K,MRR,NDCG@K_slice_a,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
0,D1_heuristic_logpop_blend,0.092892,0.19328,0.064321,0.067059,0.068322,0.204632,0.501587,6.272161,0.720097
1,D6a_rank_head,0.079514,0.16544,0.054695,0.056830,0.052844,0.208822,0.457143,6.320686,0.732713
2,D6b_rank_biencoder,0.077646,0.16144,0.053708,0.055946,0.054507,0.211762,0.787302,6.508531,0.766363
3,two_tower_v1_bare,0.018161,0.04680,0.010325,0.011008,0.020537,0.261639,0.987302,12.167556,0.995623



recs_018 — errors hit so far
----------------------------
1) load_frozen_trunk: Dense has no output_shape (D6a cell)
   Meaning: Keras 3 Dense layers are not built until first call, so .output_shape is missing.
   Why it failed: code read projection dim from an unbuilt layer.
   Fix: use model._projection_dim (or item_vectors.shape[1]). Already in ranker_d6_rank_head.py.

2) build_rank_biencoder_model: "expecting one weight" (D6b cell)
   Meaning: set_weights got a full checkpoint but the new model only had one weight slot.
   Why it failed: warm-start ran before build(); only item_base_embeddings existed.
   Fix: call rank_model.build() before set_weights(). Already in ranker_d6_biencoder.py.

3) D6b GPU OOM after D6a (D6b cell)
   Meaning: GPU ran out of memory (~5 GB already in use).
   Why it failed: D6a leaves the full retrieve model in ctx on GPU; D6b loads a second copy (+ brief third during warm-start).
   Fix: before D6b, run `del ctx, d6a_head; tf.keras.backend.clear_session